# Task 2.1 — Regularization Strength: Ridge & Lasso

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score

df = pd.read_excel('Practice_Dataset.xlsx')

# Regression target: monthly_salary
features = ['punch_count', 'hours_worked', 'satisfaction_score']
df_model = df[features + ['monthly_salary']].copy()

# Impute missing values (median) so downstream steps don't break on NaNs
imputer = SimpleImputer(strategy='median')
df_model[features + ['monthly_salary']] = imputer.fit_transform(df_model[features + ['monthly_salary']])

X = df_model[features]
y = df_model['monthly_salary']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

alphas = [0.01, 0.1, 1, 10, 50, 100]
print('alpha | Ridge R2 (train) | Ridge R2 (test)')
for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, y_train)
    train_r2 = r2_score(y_train, ridge.predict(X_train))
    test_r2 = r2_score(y_test, ridge.predict(X_test))
    print(f'{a:>6} | {train_r2:.3f} | {test_r2:.3f}')
# As alpha increases: training score drops (more constrained),
# but test score may improve if the model was overfitting

# Task 2.2 — Regularizing Tree-Based Models

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Trees overfit through depth and complexity, not an 'alpha' term.
# Regularize by limiting: max_depth, min_samples_leaf, max_features
configs = [
    {'max_depth': None, 'min_samples_leaf': 1, 'label': 'Unconstrained (overfits)'},
    {'max_depth': 8, 'min_samples_leaf': 1, 'label': 'max_depth=8'},
    {'max_depth': 8, 'min_samples_leaf': 5, 'label': 'max_depth=8, min_samples_leaf=5'},
    {'max_depth': 4, 'min_samples_leaf': 10, 'label': 'Heavily constrained'},
]
for cfg in configs:
    rf = RandomForestRegressor(n_estimators=100, max_depth=cfg['max_depth'],
                                min_samples_leaf=cfg['min_samples_leaf'], random_state=42)
    rf.fit(X_train, y_train)
    train_r2 = r2_score(y_train, rf.predict(X_train))
    test_r2 = r2_score(y_test, rf.predict(X_test))
    gap = train_r2 - test_r2
    print(f"{cfg['label']:<35} train={train_r2:.3f} test={test_r2:.3f} gap={gap:.3f}")

#  Task 2.3 — Before/After Learning Curve

In [ ]:
from sklearn.model_selection import learning_curve

# Re-plot the learning curve for the BEST regularized Random Forest
# and compare visually to the unconstrained version from Day 1
best_rf = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=5, random_state=42)
train_sizes, train_scores, val_scores = learning_curve(
    best_rf, X, y, cv=5, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 8), random_state=42
)
plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#0D9488', label='Training score')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#EF4444', label='Validation score')
plt.title('Random Forest — After Regularization')
plt.xlabel('Training Set Size')
plt.ylabel('R2 Score')
plt.legend()
plt.tight_layout()
plt.savefig(' learning_curve_after_regularization.png', dpi=150)
plt.show()